In [ ]:
from pathlib import Path
from typing import Dict

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
from IPython.display import HTML
import matplotlib.animation as animation
from scipy import stats

from neuralhydrology.datasetzoo import get_dataset, camelsus
from neuralhydrology.datautils.utils import load_scaler
from neuralhydrology.modelzoo.cudalstm import CudaLSTM
from neuralhydrology.modelzoo.shm import SHM
from neuralhydrology.nh_run import start_run, eval_run
from neuralhydrology.utils.config import Config
from neuralhydrology.training.basetrainer import BaseTrainer
from neuralhydrology.modelzoo.cfe_modules import get_pet

# What has Anna been doing?


<img src="shm.png" width="400" height="300"> <img src="lstm.png" width="400" height="300">


## Goal: LSTM + SHM = better output and state information

The goal is to assimilate the basin outflow output from an LSTM with the conceptual model SHM to hopfully get a better estimate of the outflow, and gain information about the internal states that may be more accurate. I read about what an LSTM is and about the very basics of a conceptual hydrology model.

## Stepping through a year of data in shm to get familiar with neuralhydrology

### Initialize models

This code:
 - loads config file for LSTM and getting DataLoader
 - creates an lstm instance with model weights from a previous training run
 - creates a shm instance

In [ ]:
config_file = Path("1_basin.yml") # I made batch size = 1 and sequence length = 730
# RUN THIS if you dont have results from a run already
# start_run(config_file=config_file, gpu=-1)
cudalstm_config = Config(config_file)
seq_len = cudalstm_config.seq_length

arid_basin = False

if arid_basin == False:
    run_dir = Path("runs/test_run_1203_222017")
    cuda_lstm = CudaLSTM(cfg=cudalstm_config)
    model_path = run_dir / 'model_epoch030.pt'
    model_weights = torch.load(str(model_path), map_location='cpu')  # load the weights from the file, creating the weight tensors on CPU
    cuda_lstm.load_state_dict(model_weights)  # set the new model's weights to the values loaded from file
else:
    run_dir = Path("runs/test_run_1004_101210")
    cuda_lstm = CudaLSTM(cfg=cudalstm_config)
    model_path = run_dir / 'model_epoch020.pt'
    model_weights = torch.load(str(model_path), map_location='cpu')  # load the weights from the file, creating the weight tensors on CPU
    cuda_lstm.load_state_dict(model_weights)  # set the new model's weights to the values loaded from file

# create shm model instance
shm_config_file = Path("shm_config.yml")
shm_config = Config(shm_config_file)
shm = SHM(cfg=shm_config)


### Get the data

- gets a data loader based on the config file
- skips the nan padding
- undoes scaling because shm uses raw values (no negative flow, precip, etc.)

In [ ]:
trainer = BaseTrainer(cfg = Config(config_file)) # In line 108 of basetrainer.py, I changed shuffle = False in data loader initialization
trainer.initialize_training()
loader = trainer.loader
itable = iter(loader)
for i in range(seq_len-1): # the data loader is padded with nans so skip those (this is probably not the most efficient but it works)
    _ = next(itable)
data_point = next(itable)

scaler = load_scaler(run_dir) # make sure this is for the right basin
# built in way to unscale make differnet config file
raw_x_d = {}
for feature, scaled in data_point['x_d'].items(): # key, value in dict
    center = float(scaler['xarray_feature_center'][feature].values)
    scale = float(scaler['xarray_feature_scale'][feature].values)
    raw_x_d[feature] = scaled * scale + center

center = float(scaler['xarray_feature_center']['QObs(mm/d)'].values)
scale = float(scaler['xarray_feature_scale']['QObs(mm/d)'].values)
raw_y = data_point['y'] * scale + center  

# Create a new dict with raw data, same structure as data_point
raw_data_point = {'x_d': raw_x_d,  'y': raw_y,    'date': data_point['date'], }

In [ ]:
cuda_lstm.eval()
with torch.no_grad():
    z = cuda_lstm(data_point) # takes in the scaled data
z_plus_scaled = z['y_hat'][0, :, 0] # outputs scaled prediction

# Rescale back to og for plotting/comparison
center = float(scaler['xarray_feature_center']['QObs(mm/d)'].values)
scale = float(scaler['xarray_feature_scale']['QObs(mm/d)'].values)
z_plus= z_plus_scaled * scale + center

plt.plot(data_point["date"][0], z_plus, label = "LSTM outflow (z+)")
plt.xticks(rotation=45)
plt.legend()

In [ ]:
shm_parameters = {
            "dd": torch.tensor(2.367, device="cpu", dtype=torch.float32),
            "f_thr": torch.tensor(41.688, device="cpu", dtype=torch.float32), # was 10
            "sumax": torch.tensor(41.964, device="cpu", dtype=torch.float32), # was 25
            "beta": torch.tensor(1.023, device="cpu", dtype=torch.float32),
            "perc": torch.tensor(0.711, device="cpu", dtype=torch.float32), # percent to interflow vs baseflow was .5
            "kf": torch.tensor(5.649, device="cpu", dtype=torch.float32), # unit is days to empty was 3
            "ki": torch.tensor(4.963, device="cpu", dtype=torch.float32), # unit is days was 5
            "kb": torch.tensor(31.498, device="cpu", dtype=torch.float32), # unit is days was 15
            }

In [ ]:
pt = raw_data_point
ys = pt["y"] #[1,seq_len,1]
x_d = pt['x_d'] # dict of forcing params mapped to [1,seq_len,1]

ss, sf, su, si, sb = shm.initialize_states(batch_size=1, device = 'cpu')
x = torch.tensor([ss,sf,su,si,sb])
x_history = np.zeros((6,seq_len)) # to plot, = [ss, sf, su, si, sb, z_minus, ]

for j in range(ys.shape[1]): #each day
    date = pt['date'][0, j]
    y = ys[0 ,j, 0] # true observed outflow not needed but could compare
    pet = get_pet.daily_pet_jensen2016(
                T_avg=(x_d['tmax(C)'][0, j,0] + x_d['tmin(C)'][0, j,0])/2,
                S_rad= x_d['srad(W/m2)'][0,j,0],
            )
    x_conceptual_timestep = np.stack([x_d["prcp(mm/day)"][0,j,0], pet, x_d["tmin(C)"][0,j,0], x_d["tmax(C)"][0,j,0]])
    x_conceptual_timestep = torch.tensor(x_conceptual_timestep, dtype = torch.float32, device = "cpu").unsqueeze(dim = 0) # size [1,4] row vector
    x[0], x[1], x[2], x[3], x[4], z_minus = shm.timestep_shm(x[0], x[1], x[2], x[3], x[4], shm_parameters, x_conceptual_timestep, device = x_conceptual_timestep.device)
    for i in range(5):
        x_history[i,j] = x[i]
    x_history[5,j] = z_minus

plt.figure(figsize=(20, 8))
plt.subplot(1, 2, 1)  # (rows, cols, index)
plt.plot(x_d["prcp(mm/day)"][0,:,0],label="precip",linewidth=2.0)
plt.plot(x_d["srad(W/m2)"][0,:,0], label = "srad")
T_avg=(x_d['tmax(C)'][0, :,0] + x_d['tmin(C)'][0, :,0])/2
plt.plot(T_avg, label="av. Temp (in C)", linewidth=2.0)
plt.title('Meteorogical Forcings')
plt.legend()

plt.subplot(1, 2, 2) 
plt.plot(x_history[0,:],label='snow',linewidth=2.0)
plt.plot(x_history[1,:],label='fast-flow',linewidth=2.0)
plt.plot(x_history[2,:],label='unsat-zone',linewidth=2.0)
plt.plot(x_history[3,:],label='si',linewidth=2.0)
plt.plot(x_history[4,:], label = "sb", linewidth=2.0)
plt.legend()
plt.title("States")
plt.show()

plt.figure(figsize=(20, 8))
plt.plot(x_history[5,:], label = "shm outflow (z-)")
plt.plot(z_plus, label = "LSTM outflow (z+)")
plt.plot(raw_data_point["y"][0,:,0], label = "Observed outflow")
plt.legend()
plt.title("Outflows")


## Kalman Fliter/Extended Kalman Filter

Kalman filter can be applied to a state space model with the generic format below

$x_{k+1} = M_{k+1}(x_k, \theta, u_{k+1}) + \eta_{k+1}$

$z_{k+1} = H_{k+1}(x_{k+1}, \theta) + \varepsilon_{k+1}$

$Q = E(\eta \eta^{\top})$

$R = E(\varepsilon \varepsilon^{\top})$

| symbol   | description  | what it is in this case |
| -------- | ------------ | ----------------------- | 
| $x$      | state (column) vector | snow storage, fast flow, unsaturated zone , interflow, baseflow|
| $M$      | nonlinear model operator | shm equaitons to update states |
| $\theta$ | time invariant model params | shm params. dd, f_thr, sumax etc. (I see you've been working on time variant versions of these)|
| $u$ | model input | meterological forcing |\
| $\eta$ | model error | vector length x |
| $z$ | observation vector | true outflow/LSTM outflow |
| $H$ | observation operator | shm equation for outflow |
| $\varepsilon$ | observation error | vector length x |
| $Q$ | model error covariance matrix | square size length x |
| $R$ | observation error variance | 1 |
| $P_k$ | state error covariance matrix | square size length x |

### Update equations in the order they are implemented

$z_{k+1}^- = H_{k+1}(x_{k+1}, \theta)$

$d_{k+1} = z_{k+1} - z^{-}_{k+1}$

$x_{k+1}^- = M_{k+1}(x_k^+, \theta, u_{k+1})$

$P_{k+1}^- = \bm{M}_{k+1} P_k^+ \bm{M}_{k+1}^{\top} + Q_{k+1}$

$K_{k+1} = \frac{P_{k+1}^- \bm{H}_{k+1}^{\top}}{\bm{H}_{k+1}P_{k+1}^- \bm{H}_{k+1}^{\top} + R_{k+1}}$

$x_{k+1}^+ = x_{k+1}^- + K_{k+1}d_{k+1}$

$P^+_{k+1} = P_{k+1}^- - K_{k+1} \bm{H}_{k+1} P^-_{k+1}$

| symbol   | description  |
| -------- | ------------ |
|   $d$   |    "shock", difference between predictied and observed outflow         |        
| $\bm{M}$ | linearized model matrix, jacobian of shm with respect to the states  |  
| $\bm{H}$ | linearized observation matrix |
| $P$ | State error covariance matrix |
| $K$ | Kalman Gain |

### What should H be?

Note: $ s_{k+1} = s^* - s^*/k = s^*(1 - 1/k)$, so

 $s^* = s_{k+1} (\frac{k}{k-1})$

$$\begin{align*} 
z_{k+1} &= H_{k+1}(x_{k+1}, \theta) + \epsilon_{k+1} \\
 &= \text{qfout + qiout + qbout}\\
 &= sf^*/k_f + si^*/k_i + sb^*/k_b \\
 &= sf_{k+1}(\frac{1}{k_f -1}) + si_{k+1}(\frac{1}{k_i -1}) +sb_{k+1}(\frac{1}{k_b -1})\\
\end{align*}$$

So the Jacobian of $H$ with respect to the states is

$$ \bm{H} = \begin{bmatrix}
0 & \frac{1}{k_f -1} & 0 & \frac{1}{k_i -1} & \frac{1}{k_b -1} 
\end{bmatrix}$$

This is constant in time if the parameters are constant.


## The code for extended kalman fitler

In [ ]:
ss, sf, su, si, sb = shm.initialize_states(batch_size=1, device = 'cpu')
x = torch.tensor([[ss],[sf],[su],[si],[sb]]) 
m_history = np.zeros((6,seq_len)) # to plot, = [ss, sf, su, si, sb, z_minus, ]
p_history = np.zeros((6,seq_len)) # = [ss+, sf+, su+, si+, sb+, z_plus, ]

P_p = 10.0*torch.eye(5) # initial error covariance matrix, guess?
Q = 90.0*torch.eye(5) # guess? Model error covariance
R = 100.0 # guess? observation error variance
H = torch.tensor([[0.0, 1/(shm_parameters["kf"]-1),0.0,1/(shm_parameters["ki"]-1),1/(shm_parameters["kb"]-1)]]) # 1 by 5
for j in range(seq_len): #each day
    pet = get_pet.daily_pet_jensen2016(
                T_avg=(x_d['tmax(C)'][0, j,0] + x_d['tmin(C)'][0, j,0])/2,
                S_rad= x_d['srad(W/m2)'][0,j,0],
            )
    x_conceptual_timestep = np.stack([x_d["prcp(mm/day)"][0,j,0], pet, x_d["tmin(C)"][0,j,0], x_d["tmax(C)"][0,j,0]])
    x_conceptual_timestep = torch.tensor(x_conceptual_timestep, dtype = torch.float32, device = "cpu").unsqueeze(dim = 0) # size [1,4] row vector
    shm_tensor = shm.timestep_shm_tensor_fxn(timestep_params=shm_parameters, x_conceptual_timestep=x_conceptual_timestep, device = "cpu")
    J = torch.autograd.functional.jacobian(shm_tensor, x.squeeze())
    M = J[0:5,:]
    #H = (J[5, :]).unsqueeze(dim = 0)
    x[0], x[1], x[2], x[3], x[4], z_m = shm.timestep_shm(x[0], x[1], x[2], x[3], x[4], shm_parameters, x_conceptual_timestep, device = x_conceptual_timestep.device)
    m_history[0:5,j] = x[0:5,0]
    m_history[5,j] = z_m
    z_p = z_plus[j] # LSTM outflow
    p_history[5,j] = z_p 
    d = z_p - z_m
    P_m = M @ P_p @ M.T + Q #eq 23)
    K = torch.div(P_m @ H.T , H @ P_m @ H.T + R) # 
    P_p = P_m - K @ H @ P_m #eq 25)
    x = x + K*d # eq 24)
    x = torch.clamp(x, 0) # better solution? 
    p_history[0:5, j] = x[0:5,0]

    

In [ ]:
#plotting
plt.figure(figsize=[16,10])
plt.plot(p_history[5,:], label = "z_plus (LSTM)")
plt.plot(m_history[5,:], label = "z_minus (SHM with DA)")
plt.plot(x_history[5,:], label = "shm outflow no DA")
plt.plot(raw_data_point["y"][0,:,0], label = "Observed outflow")
plt.plot()
plt.legend()
plt.show()

plt.figure(figsize = [12,8])

# Non-assimilated states with solid lines
plt.plot(x_history[0,:], label='snow storage', linewidth=2.0, color='C0')
plt.plot(x_history[1,:], label='fast-flow', linewidth=2.0, color='C1')
plt.plot(x_history[2,:], label='unsat-zone', linewidth=2.0, color='C2')
plt.plot(x_history[3,:], label='interflow', linewidth=2.0, color='C3')
plt.plot(x_history[4,:], label='baseflow', linewidth=2.0, color='C4')

# Assimilated states with dashed lines in same colors
plt.plot(p_history[0,:], label='snow storage (DA)', linewidth=2.0, color='C0', linestyle='--')
plt.plot(p_history[1,:], label='fast-flow (DA)', linewidth=2.0, color='C1', linestyle='--')
plt.plot(p_history[2,:], label='unsat-zone (DA)', linewidth=2.0, color='C2', linestyle='--')
plt.plot(p_history[3,:], label='interflow (DA)', linewidth=2.0, color='C3', linestyle='--')
plt.plot(p_history[4,:], label='baseflow (DA)', linewidth=2.0, color='C4', linestyle='--')

plt.title("State vector assimilation vs no assimilation")
plt.legend()
plt.show()

## Questions I still have:

- What are the best parameters for shm? How much do they change from basin to basin?
- Is it ok that Q is a diagonal matrix?
- ***Should H be the way it is or since the observations are from the lstm is that not correct? I think the better way to get H is use the last row of the autograd jacobian. The one I derived makes less sense becasue the ss and su elements are allways 0. The da notes also have a differnt definition of the state space model where H depends on x_t instead of x_t+1, that could just be noation or an acutal difference, idk?*** 
- How do I know if error is from bad parameters or wrong Q,R etc.
- Inequality consratints for states (no negative water)
- Can we compare the states from KF to observed data for snow storage or soil moisture?

## Code template for ensemble Kalman filter

Adding noise to the forcing (temperature, precipitation, solar radiation)

In [ ]:
x = torch.tensor([ss,sf,su,si,sb]) # 
m_history = np.zeros((6,seq_len)) # to plot, = [ss, sf, su, si, sb, z_minus, ]
p_history = np.zeros((7,seq_len)) # = [ss+, sf+, su+, si+, sb+, z_plus, sigma z]

P_p = 10.0*torch.eye(5) # initial error covariance matrix, guess?
R = 10.0 # guess? observation error covariance

N = torch.tensor(100, dtype = torch.int) # number of ensemble members
Xpts = x.unsqueeze(0).repeat(N,1) # shape [N,5]
Zpts_history = {}

for j in range(seq_len): #each day
    sigma_temp = 1 # tune?
    temp_noise = torch.randn(size = (N,))
    tmax = x_d['tmax(C)'][0,j,0] + temp_noise*sigma_temp # shape N
    tmin = x_d['tmin(C)'][0,j,0] + temp_noise*sigma_temp # shape N
    sigma_srad = 10 # tune?
    srad = x_d["srad(W/m2)"][0,j,0] + torch.randn(size = (N,))*sigma_srad
    pet = get_pet.daily_pet_jensen2016(
                T_avg=(x_d['tmax(C)'][0, j,0] + x_d['tmin(C)'][0, j,0])/2,
                S_rad= srad,
            )
    #prcp = x_d["prcp(mm/day)"][0,j,0] + torch.randn(size)
    x_conceptual_timestep = np.stack([x_d["prcp(mm/day)"][0,j,0].expand(N), pet, tmin, tmax])
    x_conceptual_timestep = torch.tensor(x_conceptual_timestep, dtype = torch.float32, device = "cpu").unsqueeze(dim = 0) # size [1,4] row vector
    
    Xpts[:,0], Xpts[:,1], Xpts[:,2], Xpts[:,3], Xpts[:,4], Zpts = shm.timestep_shm(Xpts[:,0], Xpts[:,1], Xpts[:,2], Xpts[:,3], Xpts[:,4], shm_parameters, x_conceptual_timestep, device = x_conceptual_timestep.device)
    x = Xpts.mean(dim=0)
    z = Zpts.mean()
    Zpts_history[j] = Zpts.detach().numpy()
    A = 1/torch.sqrt(N) * (Xpts - x) # scaled state ensemble perturbation matrix N,5
    V = 1/torch.sqrt(N) * (Zpts - z) # scaled output ensemble perturbation matrix N,1 
    K = A.T @ V.T * 1/(V @ V.T + R) # 5,N @ N,1  = 5,1
    Xpts = Xpts.T + K @ (z - Zpts) # 5,N = N,5 + 5,1 @ 1,N  
    Xpts = Xpts.T # back to #N,5
    xplus = Xpts.mean(dim=0)

    p_history[5,j] = z 
    x = torch.clamp(xplus, 0) # better solution? 
    p_history[0:5, j] = x[0:5]
    p_history[6,j] = torch.std(Zpts)
    


In [ ]:
# Build animation histograms
fig, ax = plt.subplots(figsize=(10, 4))

def update(j):
    ax.clear()
    zpts_data = Zpts_history[j][0]
    if np.std(zpts_data) < 1e-3:  # ensemble hasn't spread yet
        ax.text(0.5, 0.5, f'No spread at j={j}', transform=ax.transAxes, ha='center')
    else:
        ax.hist(zpts_data, bins=20)
        ax.set(xlim = (0,15))
        ax.axvline(np.mean(zpts_data), color='red', linestyle='--', linewidth=2, label=f'Mean')
        ax.axvline(np.median(zpts_data), color='orange', linestyle='--', linewidth=2, label=f'Median')
        ax.legend()
        ax.set_title(f't={j}')

ani = animation.FuncAnimation(fig, update, range(2,seq_len), interval=30)
plt.close()
HTML(ani.to_jshtml())

In [ ]:
with open('ensemble_animationQQ.html', 'w') as f:
    f.write(ani.to_jshtml())

In [ ]:
# Build animation qq plots
fig, ax = plt.subplots(figsize=(10, 4))

def updateq(j):
    ax.clear()
    zpts_data = Zpts_history[j][0]
    if np.std(zpts_data) < 1e-3:  # ensemble hasn't spread yet
        ax.text(0.5, 0.5, f'No spread at j={j}', transform=ax.transAxes, ha='center')
    else:
        stats.probplot(zpts_data, dist="norm", plot=ax)
        ax.set_title(f'Q-Q Plot: t = {j}')

ani = animation.FuncAnimation(fig, updateq, range(2,seq_len), interval=30)
plt.close()
HTML(ani.to_jshtml())

In [ ]:
# Plotting ensemble Kalman filter results
fig, ax = plt.subplots(figsize=[20, 10])

ax.plot(z_plus, label='LSTM outflow', linewidth=2)
ax.plot(p_history[5,:], label='EnKF ensemble mean outflow', linewidth=2.5, color='C1')
ax.fill_between(range(seq_len), 
                  p_history[5,:] - p_history[6,:], 
                  p_history[5,:] + p_history[6,:], 
                  alpha=0.3, color='C1', label='±1 std dev (ensemble)')
ax.plot(x_history[5,:], label='SHM outflow (no DA)', linewidth=1.5, color='C2', linestyle='--')

ax.plot(raw_data_point["y"][0,:,0], label='Observed outflow', linewidth=1, color='r')
ax.legend(fontsize=12)
plt.show()
